<a href="https://colab.research.google.com/github/mmomayck/GeoAI/blob/main/%E0%B8%A3%E0%B8%A7%E0%B8%A1%E0%B9%84%E0%B8%9F%E0%B8%A5%E0%B9%8CBurned_Area_Sentinel_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Download the dataset (image and building footprints) from Zenodo
!mkdir Dataset
!wget  https://zenodo.org/records/16948350/files/23728930_15.tiff?download=1  -O Dataset/Image.tif
!wget  https://zenodo.org/records/16948350/files/BuildingFootPrint.gpkg?download=1 -O Dataset/GT.gpkg

# Install necessary packages
!pip install rasterio
!pip install dbfread

mkdir: cannot create directory ‘Dataset’: File exists
--2026-06-09 19:02:45--  https://zenodo.org/records/16948350/files/23728930_15.tiff?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.184.103.118, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6765878 (6.5M) [image/tiff]
Saving to: ‘Dataset/Image.tif’

Dataset/Image.tif   100%[===================>]   6.45M  1.60MB/s    in 5.0s    

2026-06-09 19:02:51 (1.30 MB/s) - ‘Dataset/Image.tif’ saved [6765878/6765878]

--2026-06-09 19:02:51--  https://zenodo.org/records/16948350/files/BuildingFootPrint.gpkg?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.184.103.118, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 958464 (936K) [application/octet-stream]
Saving to: ‘Dataset/GT.gpkg’

Dataset/GT.gpkg     100%[==

In [ ]:
# Import the libraries
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio import features
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject, Resampling
import geopandas as gpd
from tqdm.notebook import tqdm
import folium
from folium import plugins

In [ ]:
import os
import glob

folder_path = '/content/drive/MyDrive/Burned_Area_Sentinel-2'

file_list = glob.glob(os.path.join(folder_path, '*'))

print(f"พบไฟล์ทั้งหมด: {len(file_list)} ไฟล์\n")


for file_path in file_list:

    file_name = os.path.basename(file_path)
    print(file_name)

พบไฟล์ทั้งหมด: 35 ไฟล์

S2C_20260423_R104_T47QMA_Com128A4.tif
S2B_20260415_R061_T47PPS_Com128A4.tfw
S2C_20260406_R004_T47QMA.shx
S2B_20260415_R061_T47PPS.shp.xml
S2B_20260105_R061_T47PQR_Com128A4.tfw
S2C_20260406_R004_T47QMA_Com128A4.tif.aux.xml
S2B_20260415_R061_T47PPS.shx
S2C_20260327_R004_T47QMA.dbf
S2C_20260327_R004_T47QMA_Com128A4.tfw
S2C_20260406_R004_T47QMA_Com128A4.tif
S2C_20260327_R004_T47QMA.sbn
S2C_20260406_R004_T47QMA_Com128A4.tif.xml
S2B_20260105_R061_T47PQR.sbx
S2B_20260105_R061_T47PQR.prj
S2B_20260105_R061_T47PQR.cpg
S2B_20260415_R061_T47PPS.cpg
S2C_20260327_R004_T47QMA.shp.xml
S2B_20260105_R061_T47PQR_Com128A4.tif.aux.xml
S2B_20260415_R061_T47PPS.dbf
S2C_20260423_R104_T47QMA.prj
S2C_20260327_R004_T47QMA.sbx
S2C_20260327_R004_T47QMA.shx
S2C_20260406_R004_T47QMA_Com128A4.tfw
S2B_20260105_R061_T47PQR.dbf
S2B_20260415_R061_T47PPS_Com128A4.tif.aux.xml
S2C_20260423_R104_T47QMA_Com128A4.tif.xml
S2C_20260406_R004_T47QMA.shp
S2B_20260105_R061_T47PQR_Com128A4.tif.xml
S2B_20260415

In [ ]:
import geopandas as gpd

shapefile_path = '/content/drive/MyDrive/Burned_Area_Sentinel-2/S2C_20260406_R004_T47QMA.shp'

gdf = gpd.read_file(shapefile_path)

gdf.head()

,geometry
0,"POLYGON ((98.22808 18.04234, 98.22808 18.04216..."
1,"POLYGON ((98.10374 18.04285, 98.10375 18.04249..."
2,"POLYGON ((98.10679 18.0425, 98.10641 18.0425, ..."
3,"POLYGON ((98.18462 18.04342, 98.18462 18.04324..."
4,"POLYGON ((98.10526 18.04359, 98.10526 18.0434,..."


In [ ]:
import rasterio
from rasterio.plot import show
import geopandas as gpd

tif_path = rasterio.open('/content/drive/MyDrive/Burned_Area_Sentinel-2/S2C_20260406_R004_T47QMA_Com128A4.tif')
img = tif_path.read()

print(f"ขนาดภาพ (Bands, Height, Width): {img.shape}")
print(f"ระบบพิกัดของภาพ (Image CRS): {tif_path.crs}")

GroundTruth = gpd.read_file("Dataset/GT.gpkg")
print(f"ระบบพิกัดของเวกเตอร์ (Vector CRS EPSG): {GroundTruth.crs.to_epsg()}")

ขนาดภาพ (Bands, Height, Width): (3, 10980, 10980)
ระบบพิกัดของภาพ (Image CRS): EPSG:32647
ระบบพิกัดของเวกเตอร์ (Vector CRS EPSG): 26986


In [ ]:
import os
import glob
import pandas as pd
import geopandas as gpd
import rasterio
from dbfread import DBF

folder_path = '/content/drive/MyDrive/Burned_Area_Sentinel-2'

all_files = glob.glob(os.path.join(folder_path, '*'))
print(f"พบไฟล์ในโฟลเดอร์ทั้งหมด: {len(all_files)} ไฟล์")

for file_path in all_files:
    file_name = os.path.basename(file_path)
    # แยกนามสกุลไฟล์เพื่อเช็คประเภท (รองรับนามสกุลซ้อน เช่น .tif.xml)
    ext = '.' + '.'.join(file_name.split('.')[1:]).lower()

    print("=" * 85)
    print(f"[FILE]: {file_name}")
    print("=" * 85)

    try:
        # --- 1. อ่านไฟล์ภาพดาวเทียม (.tif) ---
        if file_name.endswith('.tif'):
            print("ประเภท: ข้อมูลภาพถ่ายดาวเทียม (Raster Dataset)")
            with rasterio.open(file_path) as src:
                print(f"   - มิติภาพ (Rows x Columns): {src.shape}")
                print(f"   - จำนวนแบนด์ (Bands): {src.count}")
                print(f"   - ระบบพิกัดอ้างอิง (CRS): {src.crs}")
                print(f"   - ขอบเขตพื้นที่ (Bounding Box): {src.bounds}")
                # สรุปค่าสถิติแบนด์แรกเพื่อดูช่วงคะแนนพิกเซล
                band1 = src.read(1)
                print(f"   - สถิติพิกเซล (Band 1): Min = {band1.min()}, Max = {band1.max()}, Mean = {band1.mean():.2f}")

        # --- 2. อ่านไฟล์พื้นที่เชิงเส้น/โพลีกอน (.shp) ---
        elif file_name.endswith('.shp'):
            print("ประเภท: ข้อมูลเวกเตอร์เชิงพื้นที่ (Vector Shapefile)")
            gdf = gpd.read_file(file_path)
            print(f"   - จำนวนแถวข้อมูล x จำนวนคอลัมน์: {gdf.shape}")
            print(f"   - ระบบพิกัดอ้างอิง (CRS): {gdf.crs}")
            print(f"   - รายชื่อตัวแปรคอลัมน์: {list(gdf.columns)}")
            print(f"   - ตัวอย่างหน้าตาข้อมูล (2 แถวแรก):")
            print(gdf.head(2).to_string(index=False))

        # --- 3. อ่านไฟล์ฐานข้อมูลคุณลักษณะ (.dbf) ---
        elif file_name.endswith('.dbf'):
            print("ประเภท: ตารางคุณลักษณะของเวกเตอร์ (Attribute Table)")
            table = DBF(file_path, load=True)
            df = pd.DataFrame(iter(table))
            print(f"   - จำนวนข้อมูลในตาราง: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์")
            print(f"   - รายชื่อหัวตาราง: {list(df.columns)}")
            print(f"   - ตัวอย่างข้อมูลภายใน (2 แถวแรก):")
            print(df.head(2).to_string(index=False))

        # --- 4. อ่านไฟล์ข้อความพิกัดควบคุมภาพ (.tfw) ---
        elif file_name.endswith('.tfw'):
            print("ประเภท: ไฟล์พิกัดควบคุมตำแหน่งรูปภาพ (World File)")
            with open(file_path, 'r') as f:
                lines = f.read().splitlines()
            print("   - ค่าสัมประสิทธิ์พิกัดภูมิศาสตร์ (6 พารามิเตอร์):")
            for idx, line in enumerate(lines):
                print(f"     Line {idx+1}: {line}")

        # --- 5. อ่านไฟล์คำอธิบายข้อมูล (.xml / .tif.xml) ---
        elif file_name.endswith('.xml'):
            print("ประเภท: ไฟล์ข้อมูลอธิบายรายละเอียด (Geospatial Metadata XML)")
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                head_lines = [next(f).strip() for _ in range(12)]
            print("   - ตัวอย่างโครงสร้างแท็ก Metadata (12 บรรทัดแรก):")
            for line in head_lines:
                if line: print(f"     {line}")

        # --- 6. ไฟล์ระบบอื่นๆ (.shx, .sbn, .sbx, .prj, .cpg, .ovr) ---
        elif any(file_name.endswith(s_ext) for s_ext in ['.shx', '.sbn', '.sbx', '.prj', '.cpg', '.ovr']):
            size_kb = os.path.getsize(file_path) / 1024
            print("ประเภท: ไฟล์โครงสร้างระบบและดัชนีภูมิสารสนเทศ (GIS Index/Config)")
            print(f"   - ขนาดไฟล์: {size_kb:.2f} KB")
            print("   - หมายเหตุ: ไฟล์นี้เป็นตัวเชื่อมโยงระบบพิกัดและดัชนีแผนที่ ตัวโค้ด GIS จะเรียกใช้อัตโนมัติ")

        else:
            size_kb = os.path.getsize(file_path) / 1024
            print(f"ประเภท: ไฟล์อื่นๆ | ขนาด: {size_kb:.2f} KB")

    except Exception as e:
        print(f"[Error]: ไม่สามารถเจาะอ่านไฟล์นี้ได้ชั่วคราวเนื่องจาก -> {e}")

    print("\n" + "-"*85 + "\n")

print("อ่านข้อมูลทุกไฟล์เสร็จสิ้นเรียบร้อยแล้ว")

พบไฟล์ในโฟลเดอร์ทั้งหมด: 35 ไฟล์
[FILE]: S2C_20260423_R104_T47QMA_Com128A4.tif
ประเภท: ข้อมูลภาพถ่ายดาวเทียม (Raster Dataset)
   - มิติภาพ (Rows x Columns): (10980, 10980)
   - จำนวนแบนด์ (Bands): 3
   - ระบบพิกัดอ้างอิง (CRS): EPSG:32647
   - ขอบเขตพื้นที่ (Bounding Box): BoundingBox(left=399960.0, bottom=1990200.0, right=509760.0, top=2100000.0)
   - สถิติพิกเซล (Band 1): Min = 0, Max = 16366, Mean = 2570.51

-------------------------------------------------------------------------------------

[FILE]: S2B_20260415_R061_T47PPS_Com128A4.tfw
ประเภท: ไฟล์พิกัดควบคุมตำแหน่งรูปภาพ (World File)
   - ค่าสัมประสิทธิ์พิกัดภูมิศาสตร์ (6 พารามิเตอร์):
     Line 1: 10.0000000000
     Line 2: 0.0000000000
     Line 3: 0.0000000000
     Line 4: -10.0000000000
     Line 5: 600005.0000000000
     Line 6: 1700035.0000000000

-------------------------------------------------------------------------------------

[FILE]: S2C_20260406_R004_T47QMA.shx
ประเภท: ไฟล์โครงสร้างระบบและดัชนีภูมิสารสนเทศ (GIS Ind

##เอาไฟล์เหมือนกันมารวมกัน

In [ ]:
shp_files = glob.glob('/content/drive/MyDrive/Burned_Area_Sentinel-2/*.shp')
gdf_list = []

for file in shp_files:
    gdf = gpd.read_file(file)
    gdf_list.append(gdf)

merged_gdf = pd.concat(gdf_list, ignore_index=True)
output_shp = '/content/drive/MyDrive/Burned_Area_Sentinel-2/alldata/Merged_Burned_Area.shp'
merged_gdf.to_file(output_shp)

print(f"รวมไฟล์ Shapefile จำนวน {len(shp_files)} ไฟล์เสร็จสิ้น")
print(f"จำนวนข้อมูลทั้งหมดหลังรวม: {len(merged_gdf)} แถว")

/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


รวมไฟล์ Shapefile จำนวน 1 ไฟล์เสร็จสิ้น
จำนวนข้อมูลทั้งหมดหลังรวม: 2332 แถว


In [ ]:
merged_gdf

,geometry
0,"POLYGON ((98.22808 18.04234, 98.22808 18.04216..."
1,"POLYGON ((98.10374 18.04285, 98.10375 18.04249..."
2,"POLYGON ((98.10679 18.0425, 98.10641 18.0425, ..."
3,"POLYGON ((98.18462 18.04342, 98.18462 18.04324..."
4,"POLYGON ((98.10526 18.04359, 98.10526 18.0434,..."
...,...
2327,"POLYGON ((98.54122 18.94549, 98.54122 18.94476..."
2328,"POLYGON ((98.25973 18.94549, 98.25973 18.94531..."
2329,"POLYGON ((98.49869 18.94575, 98.4987 18.94503,..."
2330,"POLYGON ((98.53797 18.94585, 98.53778 18.94585..."


In [ ]:
merged_gdf = merged_gdf.set_crs('EPSG:4326', allow_override=True)

merged_gdf = merged_gdf.to_crs('EPSG:32647')

print(f"ระบบพิกัด พื้นที่เผาไหม้: {merged_gdf.crs}")

ระบบพิกัด พื้นที่เผาไหม้: EPSG:32647
